# Feature Extraction with EgoVLP + Supervised Error Recognition (V1/V2)

This notebook:
1. Clones the required repos (EgoVLP, CaptainCook4D feature_extractors, our project).
2. Mounts Google Drive and loads the EgoVLP checkpoint + the resized CaptainCook4D videos.
3. Extracts per-second (1s) EgoVLP video features for every recording and saves them to Drive with the CaptainCook4D `.npz` naming convention.
4. Symlinks the features into `code/data/video/egovlp/` so `CaptainCookStepDataset` finds them.
5. Trains the MLP (V1) and Transformer (V2) baselines on top of EgoVLP features. Checkpoints and stats are saved to Drive.

In [ ]:
# Step 1: Clone repositories
!git clone --recursive https://github.com/showlab/EgoVLP /content/egovlp
!git clone --recursive https://github.com/Laio95/aml-2025-mistake-detection.git /content/code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Dependencies: our repo + EgoVLP extras (einops, decord, timm, transformers, pytorchvideo)
!pip install -r /content/code/requirements.txt
!pip install -q einops timm transformers decord pytorchvideo
!pip install -q torcheval tqdm natsort opencv-python pillow scikit-learn

In [ ]:
# EgoVLP's FrozenInTime constructor loads a local ViT-B/16 checkpoint from
# /content/egovlp/pretrained/jx_vit_base_p16_224-80ecf9dd.pth. Download it.
!mkdir -p /content/egovlp/pretrained
!wget -q -nc -O /content/egovlp/pretrained/jx_vit_base_p16_224-80ecf9dd.pth \
  https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
!ls -la /content/egovlp/pretrained/

In [ ]:
# User-configurable paths on Drive
import os

DRIVE_ROOT = '/content/drive/MyDrive/AML_Project'
EGOVLP_CKPT = f'{DRIVE_ROOT}/models/egovlp.pth'
VIDEOS_DIR  = f'{DRIVE_ROOT}/CaptainCook4D/captain_cook_4d_gopro_resized_extracted'
FEATURES_DIR = f'{DRIVE_ROOT}/CaptainCook4D/features_egovlp'
CKPT_DIR    = f'{DRIVE_ROOT}/models'

for p in [FEATURES_DIR, CKPT_DIR]:
    os.makedirs(p, exist_ok=True)

print('EGOVLP_CKPT exists:', os.path.exists(EGOVLP_CKPT))
print('VIDEOS_DIR  exists:', os.path.exists(VIDEOS_DIR))

# Export as env vars for bash/! cells
os.environ['EGOVLP_CKPT'] = EGOVLP_CKPT
os.environ['VIDEOS_DIR'] = VIDEOS_DIR
os.environ['FEATURES_DIR'] = FEATURES_DIR
os.environ['CKPT_DIR'] = CKPT_DIR

In [ ]:
# Step 2: Extract EgoVLP features for every recording.
# Adapted from feature_extractors/frame_features/extract_features.py.
# Output: <FEATURES_DIR>/<recording_id>_360p.mp4_1s_1s.npz (shape [N, 256]).
%cd /content/code
!python segment_feature_extractor.py \
    --egovlp_repo /content/egovlp \
    --egovlp_ckpt "$EGOVLP_CKPT" \
    --videos_dir "$VIDEOS_DIR" \
    --output_dir "$FEATURES_DIR" \
    --backbone egovlp \
    --use_decord \
    --batch_size 16

In [ ]:
# Step 3: Expose the Drive features to CaptainCookStepDataset.
# The dataset reads from: <segment_features_directory>/video/<backbone>/<recording_id>_360p.mp4_1s_1s.npz
# Default segment_features_directory is "data/". We symlink data/video/egovlp -> Drive features dir.
import os, shutil

os.chdir('/content/code')
os.makedirs('data/video', exist_ok=True)
link_path = 'data/video/egovlp'
if os.path.islink(link_path) or os.path.exists(link_path):
    if os.path.islink(link_path):
        os.unlink(link_path)
    else:
        shutil.rmtree(link_path)
os.symlink(FEATURES_DIR, link_path)
print('Linked:', link_path, '->', os.readlink(link_path))
print('Sample files:', sorted(os.listdir(link_path))[:3])

In [ ]:
!wandb login

## Training the baselines on EgoVLP features

* **V1 (MLP)**: `--variant MLP`
* **V2 (Transformer)**: `--variant Transformer`

Checkpoints are saved under `$CKPT_DIR/error_recognition/<variant>/egovlp/`. Per-epoch training stats are written to `stats/error_recognition/<variant>/egovlp/*_training_performance.txt` and copied back to Drive at the end of each cell.

In [ ]:
%%bash
cd /content/code
python train_er.py \
    --num_epochs 50 --batch_size 1 --lr 1e-5 \
    --backbone egovlp --variant Transformer --split step \
    --ckpt_directory "$CKPT_DIR"
mkdir -p "$STATS_DIR"
cp -r stats/* "$STATS_DIR"/ 2>/dev/null || true
cp -r results/* /content/drive/MyDrive/AML_Project/results/ 2>/dev/null || true

In [ ]:
%%bash
cd /content/code
python train_er.py \
    --num_epochs 50 --batch_size 1 --lr 1e-4 \
    --backbone egovlp --variant MLP --split step \
    --ckpt_directory "$CKPT_DIR"
mkdir -p "$STATS_DIR"
cp -r stats/* "$STATS_DIR"/ 2>/dev/null || true
cp -r results/* /content/drive/MyDrive/AML_Project/results/ 2>/dev/null || true